# Settimana 9 — De Moivre, Laplace e la campana delle somme: il teorema del limite centrale

Questo è il notebook della Settimana 9 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_09/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## Laboratorio: la campana delle somme

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_09/#lab-somme)

In [ ]:
from math import comb, factorial, sqrt
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np

Z = NormalDist()   # normale standard: Z.cdf(z) è la funzione Phi(z)


def cdf_uniformi(x, n):
    """P(S_n <= x) esatta per la somma di n uniformi tra 0 e 1."""
    tot = sum((-1) ** k * comb(n, k) * (x - k) ** n
              for k in range(int(x) + 1))
    return tot / factorial(n)


# P(S_n < n/3): valore esatto e normale con media n/2, varianza n/12.
print("  n   esatta   normale")
for n in [2, 5, 10]:
    esatta = cdf_uniformi(n / 3, n)
    normale = Z.cdf((n / 3 - n / 2) / sqrt(n / 12))
    print(f"{n:3d}   {esatta:.4f}   {normale:.4f}")

# Dieci dadi: la distribuzione esatta della somma si costruisce
# aggiungendo un dado alla volta (np.convolve somma su tutti i modi).
dado = np.ones(6) / 6
somma = np.array([1.0])
for _ in range(10):
    somma = np.convolve(somma, dado)
valori = np.arange(10, 61)               # la somma va da 10 a 60
vince = (valori <= 25) | (valori >= 45)
mu, sigma = 35, sqrt(10 * 35 / 12)
senza = 2 * (1 - Z.cdf(10 / sigma))      # soglie 25 e 45
con = 2 * (1 - Z.cdf(9.5 / sigma))       # soglie 25,5 e 44,5
print(f"\nDieci dadi: esatta {somma[vince].sum():.4f}   "
      f"normale {senza:.4f}   con la correzione {con:.4f}")

# Istogrammi della somma di n uniformi, con la campana sovrapposta.
rng = np.random.default_rng(1733)
fig, assi = plt.subplots(1, 4, figsize=(12, 3))
for asse, n in zip(assi, [1, 2, 5, 10]):
    s = rng.random((50_000, n)).sum(axis=1)
    campana = NormalDist(n / 2, sqrt(n / 12))
    x = np.linspace(s.min(), s.max(), 200)
    asse.hist(s, bins=50, density=True, alpha=0.6)
    asse.plot(x, [campana.pdf(v) for v in x])
    asse.set_title(f"n = {n}")
plt.tight_layout()
plt.show()